# Modules and classes

In [1]:
# Numerical & data handling
import numpy as np
import pandas as pd
import math

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import LabelEncoder, PowerTransformer, StandardScaler, RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Specialized packages
from ISLP import confusion_table

# Function

In [2]:
def evaluate_models(models, X_train, y_train, X_test, y_test, model_names=None, label_encoder=None):
    """
    Train and evaluate multiple models, return summary DataFrame of classification metrics.
    
    Parameters:
    - models: list of instantiated sklearn/xgboost classifiers
    - X_train, y_train: training data
    - X_test, y_test: test data
    - model_names: optional list of names for models
    - label_encoder: optional LabelEncoder if needed for XGB
    
    Returns:
    - pd.DataFrame with accuracy, weighted precision, recall, F1
    """
    reports = []
    names = model_names if model_names else [type(m).__name__ for m in models]
    
    for model, name in zip(models, names):
        if isinstance(model, XGBClassifier):
            # Encode labels for XGB
            y_train_enc = label_encoder.fit_transform(y_train)
            y_test_enc = label_encoder.transform(y_test)
            model.fit(X_train, y_train_enc)
            y_pred_enc = model.predict(X_test)
            y_pred = label_encoder.inverse_transform(y_pred_enc)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        reports.append({
            "Model": name,
            "Accuracy": report.get("accuracy", None),
            "Weighted Precision": report["weighted avg"]["precision"],
            "Weighted Recall": report["weighted avg"]["recall"],
            "Weighted F1": report["weighted avg"]["f1-score"]
        })
    
    return pd.DataFrame(reports)


# Data loading and model setup

In [3]:
# Using the extracted file to be used in the no clustering part
X_tr_sc = pd.read_csv("X_tr_sc.csv", index_col = 0)
X_ts_sc = pd.read_csv("X_ts_sc.csv", index_col = 0)
y_train = pd.read_csv("y_train.csv", index_col = 0)
y_test = pd.read_csv("y_test.csv", index_col = 0)

# Using the extracted file to be used in clustering.
tr_sc_cl = pd.read_csv("k_2_tr_sc_cl.csv", index_col= 0)
ts_sc_cl = pd.read_csv("k_2_ts_sc_cl.csv", index_col= 0)
tr_sc_cl1 = pd.read_csv("k_2_tr_sc_cl1.csv", index_col= 0)
tr_sc_cl2 = pd.read_csv("k_2_tr_sc_cl2.csv", index_col= 0)
ts_sc_cl1 = pd.read_csv("k_2_ts_sc_cl1.csv", index_col= 0)
ts_sc_cl2 = pd.read_csv("k_2_ts_sc_cl2.csv", index_col= 0) # All obs belong to cluster 2 (index 1)

tr_sc_cl = tr_sc_cl.drop("cluster", axis = 1)
ts_sc_cl = ts_sc_cl.drop("cluster", axis = 1)
tr_sc_cl1 = tr_sc_cl1.drop("cluster", axis = 1)
tr_sc_cl2 = tr_sc_cl2.drop("cluster", axis = 1)
ts_sc_cl1 = ts_sc_cl1.drop("cluster", axis = 1)
ts_sc_cl2 = ts_sc_cl2.drop("cluster", axis = 1)

X_tr_sc_cl1 = tr_sc_cl1.iloc[:, :-1]
y_tr_cl1 = tr_sc_cl1.iloc[:, -1]
X_tr_sc_cl2 = tr_sc_cl2.iloc[:, :-1]
y_tr_cl2 = tr_sc_cl2.iloc[:, -1]
X_ts_sc_cl1 = ts_sc_cl1.iloc[:, :-1]
y_ts_cl1 = ts_sc_cl1.iloc[:, -1]
X_ts_sc_cl2 = ts_sc_cl2.iloc[:, :-1]
y_ts_cl2 = ts_sc_cl2.iloc[:, -1]

In [4]:
# Using the selected predictors from the backward elimination
selected_cols = [
    'total liabilities', 'share capital', 'total capital', 'finance costs',
    'current ratio', 'quick ratio', 'interest expense ratio (B)',
    'total liabilities/total net worth', 'operating profit/paid-in capital ratio',
    'retention ratio'
    ]

In [5]:
# Writing the parameters for the pipeline function.

models = [
    LogisticRegression(penalty= 'l2', solver= 'saga', max_iter= 10000, random_state= 1),
    DecisionTreeClassifier(random_state=1),
    RandomForestClassifier(n_estimators=500, random_state=1),
    XGBClassifier(n_estimators=500, random_state=1, eval_metric="logloss")
]
model_names = ["Multinomial Logistic Regression", "CART", "RandomForest", "XGBoost"]

In [ ]:
# XGBoost needs its categorical response variable to be encoded
le = LabelEncoder()

# Proposed model

We are going to propose 1 classification technique, i.e., K-means with 2 clusters, and 3 classification techniques, i.e., Classification and Regression Trees (CART), Random Forest (RF), and XGBoost (Extreme Gradient Boosting). We are going to compare them to our benchmark model, Multinomial Logistic Regression.

## Proposed model: No cluster

In [7]:
# Performance metrics computed without cluster-based grouping

summary_no_cluster = evaluate_models(models, 
                                     X_tr_sc[selected_cols], y_train, 
                                     X_ts_sc[selected_cols], y_test, 
                                     model_names=model_names,
                                     label_encoder= le)
print(summary_no_cluster.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.543533            0.537921         0.543533     0.534785
                           CART  0.479290            0.483352         0.479290     0.481030
                   RandomForest  0.618766            0.619810         0.618766     0.618153
                        XGBoost  0.630600            0.633047         0.630600     0.631074


## Proposed model: Cluster 1

In [8]:
# Performance metrics computed for observations in Cluster 1

summary_cl1 = evaluate_models(models, 
                              X_tr_sc_cl1[selected_cols], y_tr_cl1, 
                              X_ts_sc_cl1[selected_cols], y_ts_cl1, 
                              model_names=model_names,
                              label_encoder= le)
print(summary_cl1.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.576087            0.573100         0.576087     0.573274
                           CART  0.489130            0.496807         0.489130     0.491765
                   RandomForest  0.600932            0.601575         0.600932     0.600192
                        XGBoost  0.633540            0.632295         0.633540     0.632224


## Proposed model: Cluster 2

In [9]:
# # Performance metrics computed for observations in Cluster 2

summary_cl2 = evaluate_models(models, 
                              X_tr_sc_cl2[selected_cols], y_tr_cl2, 
                              X_ts_sc_cl2[selected_cols], y_ts_cl2, 
                              model_names=model_names,
                              label_encoder= le)
print(summary_cl2.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.528757            0.523836         0.528757     0.506887
                           CART  0.491651            0.498395         0.491651     0.494425
                   RandomForest  0.614100            0.620334         0.614100     0.612280
                        XGBoost  0.619666            0.618919         0.619666     0.617531


c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

# Summary

In [10]:
# Summary of the results

# Add a column to identify the grouping
summary_no_cluster["Group"] = "No cluster"
summary_cl1["Group"] = "Cluster 1"
summary_cl2["Group"] = "Cluster 2"

# Concatenate into one DataFrame
combined_summary = pd.concat([summary_no_cluster, summary_cl1, summary_cl2],
                             axis=0, ignore_index=True)

# Reorder columns so "Group" comes first
combined_summary = combined_summary[["Group", "Model", "Accuracy", 
                                     "Weighted Precision", "Weighted Recall", "Weighted F1"]].round(2)

print(combined_summary.to_string(index=False))


     Group                           Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
No cluster Multinomial Logistic Regression      0.54                0.54             0.54         0.53
No cluster                            CART      0.48                0.48             0.48         0.48
No cluster                    RandomForest      0.62                0.62             0.62         0.62
No cluster                         XGBoost      0.63                0.63             0.63         0.63
 Cluster 1 Multinomial Logistic Regression      0.58                0.57             0.58         0.57
 Cluster 1                            CART      0.49                0.50             0.49         0.49
 Cluster 1                    RandomForest      0.60                0.60             0.60         0.60
 Cluster 1                         XGBoost      0.63                0.63             0.63         0.63
 Cluster 2 Multinomial Logistic Regression      0.53                0.52 